# 02: Data Cleaning Pipeline

This notebook documents the full cleaning pipeline applied to the raw Diabetes 130-US Hospitals dataset. Every transformation is explained and justified below.

## Setup

In [1]:
import pandas as pd
import numpy as np

## Load Raw Data

In [2]:
df = pd.read_csv('../data/raw/diabetic_data.csv')
print(f'Raw Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

Raw Dataset: 101,766 rows x 50 columns


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Step 1: Standardize Missing Values
The raw dataset uses `?` to represent missing data across multiple columns. These are converted to standard NaN values and then imputed with meaningful labels.

In [3]:
df.replace('?', np.nan, inplace=True)

missing_report = df.isnull().sum()
missing_report = missing_report[missing_report > 0].sort_values(ascending=False)
missing_pct = (missing_report / len(df) * 100).round(1)
pd.DataFrame({'Missing Count': missing_report, 'Missing %': missing_pct})

,Missing Count,Missing %
weight,98569,96.9
max_glu_serum,96420,94.7
A1Cresult,84748,83.3
medical_specialty,49949,49.1
payer_code,40256,39.6
race,2273,2.2
diag_3,1423,1.4
diag_2,358,0.4
diag_1,21,0.0


## Step 2: Remove Invalid Records
Only 3 records have `Unknown/Invalid` gender. These are removed to maintain demographic integrity.

In [4]:
invalid_count = (df['gender'] == 'Unknown/Invalid').sum()
print(f'Records with Unknown/Invalid gender: {invalid_count}')
df = df[df['gender'] != 'Unknown/Invalid']

Records with Unknown/Invalid gender: 3


## Step 3: Drop Constant Columns
Columns with only one unique value provide zero analytical information and are removed.

In [5]:
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f'Constant columns dropped: {constant_cols}')
df.drop(columns=constant_cols, inplace=True)

Constant columns dropped: ['examide', 'citoglipton']


## Step 4: Impute Missing Values
Instead of dropping columns with missing data, we preserve them with explicit imputation labels so the full feature set remains available for analysis.

In [6]:
df['race'] = df['race'].fillna('Unknown')
df['weight'] = df['weight'].fillna('Not Recorded')
df['payer_code'] = df['payer_code'].fillna('Unknown')
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['diag_1'] = df['diag_1'].fillna('Unknown')
df['diag_2'] = df['diag_2'].fillna('Unknown')
df['diag_3'] = df['diag_3'].fillna('Unknown')
df['max_glu_serum'] = df['max_glu_serum'].fillna('Not Tested')
df['A1Cresult'] = df['A1Cresult'].fillna('Not Tested')

print(f'Remaining nulls: {df.isnull().sum().sum()}')

Remaining nulls: 0


## Step 5: Age Standardization
The age column contains categorical intervals like `[50-60)`. Converting these to numerical midpoints enables correlation analysis with readmission outcomes.

In [7]:
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)
df['age'].value_counts().sort_index()

age
5       161
15      691
25     1657
35     3775
45     9685
55    17256
65    22482
75    26066
85    17197
95     2793
Name: count, dtype: int64

## Step 6: Map Clinical IDs to Readable Labels
The raw dataset uses numeric IDs for admission type, discharge disposition, and admission source. Mapping these to clinical labels makes the data interpretable for dashboards.

In [8]:
admission_type_map = {
    1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4: 'Newborn',
    5: 'Not Available', 6: 'Not Available', 7: 'Trauma Center', 8: 'Not Available'
}
df['admission_type_id'] = df['admission_type_id'].map(admission_type_map).fillna('Other')

discharge_map = {
    1: 'Discharged to Home', 2: 'Short-term Hospital Transfer',
    3: 'SNF Transfer', 4: 'ICF Transfer', 5: 'Another Facility Transfer',
    6: 'Home Health Service', 7: 'AMA (Left Against Advice)',
    8: 'Home Health Service', 9: 'Admitted as Inpatient',
    10: 'Neonate Transfer', 11: 'Expired', 12: 'Still Patient',
    13: 'Hospice/Home', 14: 'Hospice/Medical Facility',
    15: 'Swing Bed', 16: 'Another Rehab Facility',
    17: 'Another Rehab Facility', 18: 'Not Available',
    19: 'Expired (Admit)', 20: 'Expired (Home)', 21: 'Expired (Medical)',
    22: 'Rehab/Long-term', 23: 'Long-term Care Hospital',
    24: 'Nursing Facility (Medicaid)', 25: 'Not Available',
    27: 'Federal Healthcare', 28: 'Psychiatric Hospital'
}
df['discharge_disposition_id'] = df['discharge_disposition_id'].map(discharge_map).fillna('Other')

admission_source_map = {
    1: 'Physician Referral', 2: 'Clinic Referral', 3: 'HMO Referral',
    4: 'Transfer from Hospital', 5: 'Transfer from SNF',
    6: 'Transfer from Another Facility', 7: 'Emergency Room',
    8: 'Court/Law Enforcement', 9: 'Not Available',
    10: 'Transfer from Critical Access', 11: 'Normal Delivery',
    13: 'Normal Delivery', 14: 'Normal Delivery',
    17: 'Not Available', 20: 'Not Available',
    22: 'Transfer from Hospital (Extramural)', 25: 'Transfer from Ambulatory Surgery'
}
df['admission_source_id'] = df['admission_source_id'].map(admission_source_map).fillna('Other')

## Step 7: Classify Diagnosis Codes
ICD9 diagnosis codes are mapped into 17 high-level clinical groups for pattern analysis.

In [9]:
def classify_icd9(code):
    if pd.isnull(code) or str(code) == 'Unknown':
        return 'Unknown'
    code = str(code).strip()
    if code.startswith('V') or code.startswith('E'):
        return 'External/Supplementary'
    try:
        val = float(code)
    except ValueError:
        return 'Unknown'
    if 250 <= val < 251: return 'Diabetes'
    elif 390 <= val <= 459 or val == 785: return 'Circulatory'
    elif 460 <= val <= 519 or val == 786: return 'Respiratory'
    elif 520 <= val <= 579 or val == 787: return 'Digestive'
    elif 580 <= val <= 629 or val == 788: return 'Genitourinary'
    elif 710 <= val <= 739: return 'Musculoskeletal'
    elif 800 <= val <= 999: return 'Injury/Poisoning'
    elif 140 <= val <= 239: return 'Neoplasms'
    elif 240 <= val <= 279: return 'Endocrine/Metabolic'
    elif 280 <= val <= 289: return 'Blood Disorders'
    elif 290 <= val <= 319: return 'Mental Disorders'
    elif 320 <= val <= 389: return 'Nervous System'
    elif 680 <= val <= 709: return 'Skin/Subcutaneous'
    elif 1 <= val <= 139: return 'Infectious/Parasitic'
    else: return 'Other'

df['diag_1_group'] = df['diag_1'].apply(classify_icd9)
df['diag_2_group'] = df['diag_2'].apply(classify_icd9)
df['diag_3_group'] = df['diag_3'].apply(classify_icd9)

df['diag_1_group'].value_counts()

diag_1_group
Circulatory               30436
Respiratory               14423
Digestive                  9475
Diabetes                   8757
Injury/Poisoning           6972
Genitourinary              5117
Musculoskeletal            4957
Other                      3951
Neoplasms                  3433
Infectious/Parasitic       2768
Endocrine/Metabolic        2702
Skin/Subcutaneous          2530
Mental Disorders           2262
External/Supplementary     1645
Nervous System             1211
Blood Disorders            1103
Unknown                      21
Name: count, dtype: int64

## Step 8: Create Readmission Target Variables
Three binary columns are created to track patients readmitted under 30 days, after 30 days, and those never readmitted. This enables multi-dimensional analysis of readmission patterns.

In [10]:
df['readmit_under_30'] = (df['readmitted'] == '<30').astype(int)
df['readmit_over_30'] = (df['readmitted'] == '>30').astype(int)
df['not_readmitted'] = (df['readmitted'] == 'NO').astype(int)

print(f"Readmitted Under 30 Days: {df['readmit_under_30'].sum():,}")
print(f"Readmitted After 30 Days: {df['readmit_over_30'].sum():,}")
print(f"Not Readmitted: {df['not_readmitted'].sum():,}")

Readmitted Under 30 Days: 11,357
Readmitted After 30 Days: 35,545
Not Readmitted: 54,861


## Step 9: Rename Columns
All columns are renamed to professional, Tableau-friendly labels and the final dataset is exported.

In [11]:
rename_map = {
    'encounter_id': 'Encounter ID', 'patient_nbr': 'Patient ID',
    'race': 'Race', 'gender': 'Gender', 'age': 'Age', 'weight': 'Weight',
    'admission_type_id': 'Admission Type',
    'discharge_disposition_id': 'Discharge Disposition',
    'admission_source_id': 'Admission Source',
    'time_in_hospital': 'Days in Hospital',
    'payer_code': 'Payer Code', 'medical_specialty': 'Medical Specialty',
    'num_lab_procedures': 'Lab Procedures',
    'num_procedures': 'Non-Lab Procedures',
    'num_medications': 'Number of Medications',
    'number_outpatient': 'Prior Outpatient Visits',
    'number_emergency': 'Prior Emergency Visits',
    'number_inpatient': 'Prior Inpatient Visits',
    'diag_1': 'Primary Diagnosis (ICD9)',
    'diag_2': 'Secondary Diagnosis (ICD9)',
    'diag_3': 'Additional Diagnosis (ICD9)',
    'diag_1_group': 'Primary Diagnosis Group',
    'diag_2_group': 'Secondary Diagnosis Group',
    'diag_3_group': 'Additional Diagnosis Group',
    'number_diagnoses': 'Total Diagnoses',
    'max_glu_serum': 'Glucose Serum Test',
    'A1Cresult': 'A1C Result',
    'change': 'Medication Changed',
    'diabetesMed': 'Diabetes Medication Prescribed',
    'readmitted': 'Readmission Status',
    'readmit_under_30': 'Readmitted Under 30 Days',
    'readmit_over_30': 'Readmitted After 30 Days',
    'not_readmitted': 'Not Readmitted',
    'metformin': 'Metformin', 'repaglinide': 'Repaglinide',
    'nateglinide': 'Nateglinide', 'chlorpropamide': 'Chlorpropamide',
    'glimepiride': 'Glimepiride', 'acetohexamide': 'Acetohexamide',
    'glipizide': 'Glipizide', 'glyburide': 'Glyburide',
    'tolbutamide': 'Tolbutamide', 'pioglitazone': 'Pioglitazone',
    'rosiglitazone': 'Rosiglitazone', 'acarbose': 'Acarbose',
    'miglitol': 'Miglitol', 'troglitazone': 'Troglitazone',
    'tolazamide': 'Tolazamide', 'insulin': 'Insulin',
    'glyburide-metformin': 'Glyburide-Metformin',
    'glipizide-metformin': 'Glipizide-Metformin',
    'glimepiride-pioglitazone': 'Glimepiride-Pioglitazone',
    'metformin-rosiglitazone': 'Metformin-Rosiglitazone',
    'metformin-pioglitazone': 'Metformin-Pioglitazone'
}
df.rename(columns=rename_map, inplace=True)



## Step 9: Preparing Medication Changes Columns

In [12]:
med_cols = ['Metformin', 'Repaglinide', 'Nateglinide', 'Chlorpropamide', 'Glimepiride',
            'Acetohexamide', 'Glipizide', 'Glyburide', 'Tolbutamide', 'Pioglitazone',
            'Rosiglitazone', 'Acarbose', 'Miglitol', 'Troglitazone', 'Tolazamide',
            'Insulin', 'Glyburide-Metformin', 'Glipizide-Metformin', 
            'Glimepiride-Pioglitazone', 'Metformin-Rosiglitazone', 'Metformin-Pioglitazone']


#We create a binary flag readmit_30d = 1 if the patient came back within 30 days, otherwise 0.
df['readmit_30d'] = df['Readmitted Under 30 Days'].astype(int)

def readmit_category(row):
    if row['Readmitted Under 30 Days'] == 1:
        return '<30 days'
    elif row['Readmitted After 30 Days'] == 1:
        return '30+ days'
    else:
        return 'Not readmitted'
df['readmit_cat'] = df.apply(readmit_category, axis=1)


#We check all medication columns: if any column is not No, then the patient was prescribed at least one diabetes medication.
def any_med_prescribed(row):
    for med in med_cols:
        if row[med] != 'No':
            return True
    return False
df['any_med_prescribed'] = df.apply(any_med_prescribed, axis=1)


# Create "Any Medication Change" flag
def any_med_change(row):
    for med in med_cols:
        if row[med] in ['Up', 'Down']:
            return 'Changed'
    if 'Medication Changed' in df.columns and row['Medication Changed'] == 'Ch':
        return 'Changed'
    return 'Steady / No change'
df['med_change_flag'] = df.apply(any_med_change, axis=1)


#For each medication, create a human‑readable change type column
for med in med_cols:
    df[f'{med}_change_type'] = df[med].apply(lambda x: 
        'Dose increased' if x == 'Up' else
        'Dose decreased' if x == 'Down' else
        'Steady dose' if x == 'Steady' else
        'Not prescribed'
    )



In [13]:
# 1. Any change vs no change
summary_any_change = df.groupby('med_change_flag').agg(
    total_patients=('readmit_30d', 'count'),
    readmit_30d_count=('readmit_30d', 'sum'),
    readmit_rate_percent=('readmit_30d', lambda x: round(100 * x.mean(), 1))
).reset_index()

# 2. By drug and change type (top 5)
top_meds = ['Metformin', 'Insulin', 'Glipizide', 'Glyburide', 'Pioglitazone']
med_summary_list = []
for med in top_meds:
    if med in df.columns:
        temp = df.groupby(f'{med}_change_type').agg(
            medication=('readmit_30d', lambda x: med),
            change_type=(f'{med}_change_type', 'first'),
            total_patients=('readmit_30d', 'count'),
            readmit_rate_percent=('readmit_30d', lambda x: round(100 * x.mean(), 1))
        ).reset_index(drop=True)
        med_summary_list.append(temp)
med_change_summary = pd.concat(med_summary_list, ignore_index=True)

# 3. By number of medication changes (count of Up/Down across all drugs)
def count_med_changes(row):
    changes = 0
    for med in med_cols:
        if row[med] in ['Up', 'Down']:
            changes += 1
    return changes
df['num_med_changes'] = df.apply(count_med_changes, axis=1)

change_count_summary = df.groupby('num_med_changes').agg(
    total_patients=('readmit_30d', 'count'),
    readmit_rate_percent=('readmit_30d', lambda x: round(100 * x.mean(), 1))
).reset_index()

# 4. Cross tabulation: med_change_flag vs readmit_cat
cross_tab = pd.crosstab(df['med_change_flag'], df['readmit_cat'], normalize='index') * 100
cross_tab = cross_tab.reset_index().rename_axis(None, axis=1)

## Step 10: Saving Output

In [ ]:
med_change_summary.to_csv('summary_by_med_change_type.csv', index=False)
change_count_summary.to_csv('summary_by_num_med_changes.csv', index=False)

print("✅ All files exported. You can now load them into Tableau.")

✅ All files exported. You can now load them into Tableau.


In [16]:
output_path = '../data/processed/cleaned.csv'
df.to_csv(output_path, index=False)

print(f'\n✅ Final Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'✅ Zero nulls: {df.isnull().sum().sum()}')
print(f'✅ Saved to: {output_path}')


✅ Final Dataset: 101,763 rows x 80 columns
✅ Zero nulls: 0
✅ Saved to: ../data/processed/cleaned.csv
